<a href="https://colab.research.google.com/github/mrivassnj-svg/HCC_ITAI_1371_SPR26/blob/main/PIG_IMPACT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This loads the dataset for the project that is geraed to serve the classification requirement
!ls -l /kaggle/input/empres-global-animal-disease-surveillance/

total 2788
-rw-r--r-- 1 1000 1000 2850933 May  6 01:36 Outbreak_240817.csv


In [ ]:
# ---------------------------
# 1. Dependencies & Data Loading (Step 5 Compliant)
# ---------------------------
import os
import pandas as pd
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Set the path to the file you'd like to load
file_path = "Outbreak_240817.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "tentotheminus9/empres-global-animal-disease-surveillance",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

# ---------------------------
# 2. Preprocessing (5 pts)
# ---------------------------
# Targeted filtering for Swine-related impacts
df_swine = df[df['speciesDescription'].str.contains('pig|swine', case=False, na=False)].copy()

# Create Binary Target: 1 if "sumCases" > 0, 0 otherwise
df_swine['Is_Target_Disease'] = (df_swine['sumCases'] > 0).astype(int)

# Drop high-cardinality/unnecessary columns
cols_to_drop = ['outbreakId', 'diseaseName', 'speciesDescription', 'source', 'localityName',
                'sumCases', 'sumDeaths', 'sumDestroyed', 'sumSlaughtered', 'humansAffected', 'humansDeaths']
df_swine = df_swine.drop(columns=[c for c in cols_to_drop if c in df_swine.columns])

# Fill missing numerical data
df_swine = df_swine.fillna(df_swine.mean(numeric_only=True))

# One-hot encode categorical features
categorical_cols = df_swine.select_dtypes(include='object').columns
if len(categorical_cols) > 0:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoded_features = encoder.fit_transform(df_swine[categorical_cols])
    encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_cols))
    df_swine = df_swine.drop(columns=categorical_cols).reset_index(drop=True)
    df_swine = pd.concat([df_swine, encoded_df], axis=1)

# ---------------------------
# 3. Data Split: 70/15/15 (Requirement)
# ---------------------------
X = df_swine.drop(columns=['Is_Target_Disease'])
y = df_swine['Is_Target_Disease']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Scale numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# ---------------------------
# 4. Individual Model Training (20 pts)
# ---------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC(probability=True, random_state=42)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

# ---------------------------
# 5. Validation & Comparison (20 pts)
# ---------------------------
def evaluate(models_dict, X_data, y_true):
    metrics = {}
    for name, model in models_dict.items():
        preds = model.predict(X_data)
        probs = model.predict_proba(X_data)[:, 1] if hasattr(model, "predict_proba") else preds
        metrics[name] = {
            "Accuracy": accuracy_score(y_true, preds),
            "Precision": precision_score(y_true, preds, zero_division=0),
            "Recall": recall_score(y_true, preds, zero_division=0),
            "F1-Score": f1_score(y_true, preds, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, probs)
        }
    return pd.DataFrame(metrics).T

val_results = evaluate(trained_models, X_val, y_val)

# ---------------------------
# 6. Ensemble & Bayesian Models (20 pts)
# ---------------------------
# Voting Ensemble (Top 3 based on F1-Score)
best_3_names = val_results.sort_values(by="F1-Score", ascending=False).head(3).index.tolist()
best_3_estimators = [(name, trained_models[name]) for name in best_3_names]

voting_model = VotingClassifier(estimators=best_3_estimators, voting='soft')
voting_model.fit(X_train, y_train)

# Bayesian Model (Gaussian Naive Bayes)
bayesian_model = GaussianNB()
bayesian_model.fit(X_train, y_train)

# Add to results
final_models = {"Ensemble (Voting)": voting_model, "Bayesian Model": bayesian_model}
test_results = evaluate({**trained_models, **final_models}, X_test, y_test)

# ---------------------------
# 7. Final Comparison Table
# ---------------------------
print("FINAL TEST METRICS COMPARISON")
display(test_results)

Using Colab cache for faster access to the 'empres-global-animal-disease-surveillance' dataset.
FINAL TEST METRICS COMPARISON


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Logistic Regression,0.971429,0.980583,0.990196,0.985366,0.954248
Decision Tree,0.980952,0.990196,0.990196,0.990196,0.828431
Random Forest,0.980952,0.980769,1.000000,0.990291,0.808824
Gradient Boosting,0.980952,0.990196,0.990196,0.990196,0.821895
KNN,0.971429,0.971429,1.000000,0.985507,0.661765
SVC,0.971429,0.971429,1.000000,0.985507,0.970588
Ensemble (Voting),0.980952,0.980769,1.000000,0.990291,0.954248
Bayesian Model,0.942857,0.989796,0.950980,0.970000,0.808824
